In [12]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [13]:
def Area_ov(rw, ydis, R_rotor):
    T1 = rw**2 * np.arccos((rw**2 - R_rotor**2 + ydis**2) / (2 * ydis * rw))
    T2 = R_rotor**2 * np.arccos((R_rotor**2 - rw**2 + ydis**2) / (2 * ydis * R_rotor))
    T3 = ((rw + R_rotor)**2 - ydis**2) * (ydis**2 - (rw - R_rotor)**2)
    ovl = T1 + T2 - 0.5 * T3**0.5
    return ovl

In [14]:


def power_turb(u_wind, a_rotor, rho_air, delta_u_s, qs):
    # Function to calculate Turbine power
    c_p = 0.5 * (1 + qs) * (1 - qs**2)  # Power Coefficient
    v = u_wind * (1 - np.sqrt(delta_u_s))  # Wind speed after the rotor disc
    P = 0.5 * (rho_air * a_rotor * v**3 * c_p)  # Turbine Power
    return v, P

In [15]:
def overlap(N_turbine, M_turbine, R_rotor, k, xs, ys):
    NM = N_turbine * M_turbine
    ovl = np.zeros((NM, NM))
    for i in range(NM):
        for j in range(NM):
            xdis = xs[i] - xs[j]
            if xdis > 0:
                rw = R_rotor + (k * abs(xdis))
                rlap = rw + R_rotor
                fullap = rw - R_rotor

                ydis = abs(ys[i] - ys[j])
                if fullap < ydis < rlap:
                    ovl[i, j] = Area_ov(rw, ydis, R_rotor)
                elif ydis <= fullap:
                    ovl[i, j] = np.pi * R_rotor**2
                else:
                    ovl[i, j] = 0
            else:
                ovl[i, j] = 0
    ovl /= np.pi * R_rotor**2
    return ovl

In [16]:
def Power_turb(U_wind, A_rotor, rho_air, delta_u_s, qs):
    C_p = 0.5 * (1 + qs) * (1 - qs**2)
    v = U_wind * (1 - np.sqrt(delta_u_s))
    P = 0.5 * (rho_air * A_rotor * v**3 * C_p)
    return v, P


In [17]:
def Velocity_def(N_turbine, M_turbine, R_rotor, k, xs, qs, ovl):
    NM = N_turbine * M_turbine
    delta_u_s = np.zeros(NM)
    for i in range(NM):
        for j in range(NM):
            delta_u_s[i] += ovl[i, j] * ((1 - qs[j]) / (1 + k * abs(xs[i] - xs[j]) / R_rotor)**2)**2
    return delta_u_s


In [18]:
def Power_sum(N_turbine, M_turbine, qs):
    NM = N_turbine * M_turbine
    q_Betz_limit = (1/3) * np.ones((N_turbine, M_turbine))
    q = q_Betz_limit
    qs = q.reshape(-1, 1)
    P = np.ones((N_turbine, M_turbine))
    Ps = P.reshape(-1, 1)
    delta_u = np.ones((N_turbine, M_turbine))
    delta_u_s = delta_u.reshape(-1, 1)
    delta_u = np.ones((N_turbine, M_turbine))
    U_wind = 10
    k = 0.04
    rho_air = 1.225
    R_rotor = 41.2
    dx = 858.562
    dy = -481.706
    angl = 45
    angl += 90
    A_rotor = 10
    x, y = Wind_farm(angl, N_turbine, M_turbine)
    xs = x.flatten()
    ys = y.flatten()
    ovl = overlap(N_turbine, M_turbine, R_rotor, k, xs, ys)
    delta_u_s = Velocity_def(N_turbine, M_turbine, R_rotor, k, xs, qs, ovl)
    v, P = Power_turb(U_wind, A_rotor, rho_air, delta_u_s, qs)
    P_sum = np.sum(P)
    return P, P_sum


In [19]:
def Wind_farm(angl, N_turbine, M_turbine):
    R_rotor = 41.2
    dx = 481.70637862320414186300933810117
    dy = 858.56241559894146317763280998799
    xR = dx / R_rotor
    yR = dy / R_rotor
    x0, y0 = 0, 0
    x = np.ones((N_turbine, M_turbine))
    y = np.ones((N_turbine, M_turbine))
    for i in range(N_turbine):
        for j in range(M_turbine):
            x[i, j] = (i - 1) * dx + np.tan(np.deg2rad(8)) * (j - 1) * dy
            y[i, j] = (j - 1) * dy + np.tan(np.deg2rad(2)) * (i - 1) * dx
    if angl != 0:
        anglr = -np.deg2rad(angl)
        for i in range(N_turbine):
            for j in range(M_turbine):
                rx = np.cos(anglr) * x[i, j] + np.sin(anglr) * y[i, j]
                ry = -np.sin(anglr) * x[i, j] + np.cos(anglr) * y[i, j]
                x[i, j] = rx
                y[i, j] = ry
    return x, y

In [20]:
def P_reverse(qs):
    N_turbine = 8
    M_turbine = 9
    NM = N_turbine * M_turbine
    Ps = np.ones(NM)
    P_sum = 1
    _, P_sum = Power_sum(N_turbine, M_turbine, qs)
    P_rev = 10000000 / P_sum
    return P_rev


In [21]:
def p_total(angl):
    N_turbine = 9
    M_turbine = 8
    NM = N_turbine * M_turbine
    q_Betz_limit = (1/3) * np.ones((N_turbine, M_turbine))
    q = q_Betz_limit
    qs = q.reshape(-1, 1)
    P = np.ones((N_turbine, M_turbine))
    Ps = P.reshape(-1, 1)
    delta_u = np.ones((N_turbine, M_turbine)).flatten()
    U_wind = 10
    k = 0.04
    rho_air = 1.225
    R_rotor = 41.2
    dx = 858.562
    dy = -481.706
    A_rotor = np.pi * R_rotor**2
    x, y = Wind_farm(angl, N_turbine, M_turbine)
    xs = x.flatten()
    ys = y.flatten()
    ovl = overlap(N_turbine, M_turbine, R_rotor, k, xs, ys)
    delta_u_s = Velocity_def(N_turbine, M_turbine, R_rotor, k, xs, qs, ovl)
    v, P = Power_turb(U_wind, A_rotor, rho_air, delta_u_s, qs)
    Power_total = np.sum(P)
    return Power_total


In [22]:
def p_tot_opt(angl):
    N_turbine = 9
    M_turbine = 8
    NM = N_turbine * M_turbine
    q_Betz_limit = (1/3) * np.ones((N_turbine, M_turbine))
    q = q_Betz_limit
    qs = q.reshape(-1, 1)
    P = np.ones((N_turbine, M_turbine))
    Ps = P.reshape(-1, 1)
    delta_u = np.ones((N_turbine, M_turbine))
    delta_u_s = delta_u.reshape(-1, 1)
    U_wind = 10  # m/sU_wind = 10  # m/s
    k = 0.04
    rho_air = 1.225  # kg/m^3
    R_rotor = 41.2  # m
    dx = 858.562
    dy = -481.706
    A_rotor = np.pi * R_rotor**2  # m^2
    x, y = Wind_farm(angl, N_turbine, M_turbine)
    xs = x.flatten()
    ys = y.flatten()
    # Calculating the overlap area
    ovl = overlap(N_turbine, M_turbine, R_rotor, k, xs, ys)
    delta_u_i = np.zeros(N_turbine)
    p_tot_new = 10
    p_tot_old = 0
    p_tot_old_2 = 0
    it = 0
    # Open file for writing
    with open('celldata2.dat', 'w') as fileID:
        while (p_tot_new - p_tot_old_2) > 0.001:
            p_tot_old_2 = p_tot_new
            it += 1
            mar = 0
            # Marching along the points
            for l in range(NM):
                p_tot_old = 0
                mar += 1

            # Iteration at one point
            while (p_tot_new - p_tot_old) > 0.001:
                p_tot_old = p_tot_new

                if qs[l] < 1:
                    qs[l] = qs[l] * 1.001

                # Calculating velocity deficit
                delta_u_s = Velocity_def(N_turbine, M_turbine, R_rotor, k, xs, qs, ovl)

                # Calculating Turbine power
                v, P = Power_turb(U_wind, A_rotor, rho_air, delta_u_s, qs)
                p_tot_new = np.sum(P)

                # Writing data to file
                fileID.write(f'{it} {mar} {p_tot_new}\n')
                # Calculate total power
                Power_total = p_tot_new

In [23]:
# Problem 2, task a: Wind Rose
P_rose = np.ones(360)
p_opt_rose = np.ones(360)
zita = np.ones(360)
ang = np.ones(360)

for i in range(360):
    ang[i] = i
    oo = i

    angl = ang[i]
    Power_total = p_total(angl)
    P_rose[i] = Power_total
    angl = ang[i]
    Power_total = p_tot_opt(angl)
    p_opt_rose[i] = Power_total
    zita[i] = (p_opt_rose[i] / P_rose[i]) - 1

P_sum = np.sum(P_rose)
P_opt_sum = np.sum(p_opt_rose)

# Plotting the wind rose
plt.figure(figsize=(8, 8))
plt.polar(np.deg2rad(ang), P_rose / (72 * 1.935559901196767e+06))
plt.title('Wind Rose')
plt.show()


C:\Users\meroe\AppData\Local\Temp\ipykernel_16472\1468546643.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  delta_u_s[i] += ovl[i, j] * ((1 - qs[j]) / (1 + k * abs(xs[i] - xs[j]) / R_rotor)**2)**2


KeyboardInterrupt: 

In [ ]:
import numpy as np

def power_sum(N_turbine, M_turbine, qs):
    # Dimension wind farm
    NM = N_turbine * M_turbine
    P = np.ones((N_turbine, M_turbine))
    P_sum = np.sum(P)

    # Free wind speed
    U_wind = 10  # [m/s]

    # Wake-decay parameter
    k = 0.04

    # Air density
    rho_air = 1.225  # [kg/m^3]

    # Rotor disc radius
    R_rotor = 41.2  # [m]

    dx = 858.562
    dy = -481.706

    # Angle calculation
    angl = 45
    angl = angl + 90

    # Rotor disc area
    A_rotor = 10  # [m^2]

    # Wind farm layout
    x, y = wind_farm(angl, N_turbine, M_turbine)
    xs = x.reshape(-1, 1)
    ys = y.reshape(-1, 1)

    # Calculating the overlap area
    ovl = overlap(N_turbine, M_turbine, R_rotor, k, xs, ys)

    # Calculating velocity deficit
    delta_u_s = velocity_def(N_turbine, M_turbine, R_rotor, k, xs, qs, ovl)

    # Calculating Turbine power
    v, P = power_turb(U_wind, A_rotor, rho_air, delta_u_s, qs)
    P_sum = np.sum(P)

    return P, P_sum




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from math import pi
# Problem 2: Wind Rose
ang = np.arange(1, 361)
P_rose = np.ones(360)
p_opt_rose = np.ones(36)
zita = np.ones(360)

for i in range(360):
    angl = ang[i]
    Power_total = p_total(angl)  # Function p_total not provided
    P_rose[i] = Power_total

    Power_total = p_tot_opt(angl)  # Function p_tot_opt not provided
    p_opt_rose[i] = Power_total

    zita[i] = (p_opt_rose[i] / P_rose[i]) - 1

P_sum = np.sum(P_rose)
P_opt_sum = np.sum(p_opt_rose)

# Plotting the wind rose
plt.figure(1, figsize=(8, 8))
plt.polar(np.deg2rad(ang), P_rose / (72 * 1.935559901196767e+06))

# Problem 2: Gain Potential of the wind farm
num = pd.read_excel('file3.xlsx', header=None)

# Plotting the potential gain
plt.figure(2, figsize=(8, 5))
plt.plot(num.iloc[:, 0], num.iloc[:, 1:])
plt.legend({'k=0.025', 'k=0.04', 'k=0.05'}, loc='northwest')
plt.xlabel('\u03B8')
plt.ylabel('\u03BE')
plt.xlim([0, 360])
plt.ylim([0, 1])

# Problem 2: Sequential Optimization (Induction factor and Power output)
N_turbine = 9
M_turbine = 8
NM = N_turbine * M_turbine
q_Betz_limit = (1 / 3) * np.ones((N_turbine, M_turbine))
q = q_Betz_limit
qs = q.flatten()
P = np.ones((N_turbine, M_turbine))
Ps = P.flatten()
delta_u = np.ones((N_turbine, M_turbine))
delta_u_s = delta_u.flatten()

U_wind = 10  # [m/s]
k = 0.04
rho_air = 1.225  # [kg/m^3]
R_rotor = 41.2  # [m]
A_rotor = np.pi * R_rotor**2

x, y = wind_farm(135, N_turbine, M_turbine)
xs = x.flatten()
ys = y.flatten()

ovl = overlap(N_turbine, M_turbine, R_rotor, k, xs, ys)
delta_u_s = velocity_def(N_turbine, M_turbine, R_rotor, k, xs, qs, ovl)
v, P = power_turb(U_wind, A_rotor, rho_air, delta_u_s, qs)

P_tot_betz = np.sum(P)
P_betz = P.flatten()

# ... (continue with the rest of the code)


ValueError: too many values to unpack (expected 1)

In [ ]:
from math import pi

# Problem 2, task a: Wind Rose
ang = np.arange(1, 361)
P_rose = np.ones(360)
p_opt_rose = np.ones(360)
zita = np.ones(360)

for i in range(360):
    angl = ang[i]
    Power_total = p_total(angl)
    P_rose[i] = Power_total
    power_total_opt = p_tot_opt(angl)
    p_opt_rose[i] = power_total_opt
    zita[i] = (p_opt_rose[i] / P_rose[i]) - 1

P_sum = np.sum(P_rose)
P_opt_sum = np.sum(p_opt_rose)

# Plotting the wind rose
plt.figure(1)
plt.polar(np.deg2rad(ang), P_rose / (72 * 1.935559901196767e+06))




            






ValueError: too many values to unpack (expected 1)

In [ ]:
# Problem 2, task a: Gain Potential of the wind farm
num = np.load('file3.xlsx', delimiter=',')

# Plotting the potential gain
plt.figure(2)
plt.plot(num[:, 0], num[:, 1:])
plt.legend(['k=0.025', 'k=0.04', 'k=0.05'], loc='northwest')
plt.xlabel('\u03B8')
plt.ylabel('\u03BE')
plt.xlim([0, 360])
plt.ylim([0, 1])



TypeError: load() got an unexpected keyword argument 'delimiter'

In [ ]:
import pandas as pd

# Load data from Excel file
df = pd.read_excel('file3.xlsx', header=None)

# Extract NumPy array
num = df.to_numpy()

# Plotting the potential gain
plt.figure(2)
plt.plot(num[:, 0], num[:, 1:])
plt.legend(['k=0.025', 'k=0.04', 'k=0.05'], loc='northwest')
plt.xlabel('\u03B8')
plt.ylabel('\u03BE')
plt.xlim([0, 360])
plt.ylim([0, 1])




ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [ ]:
# Problem 2, task a: Sequential Optimization (Induction factor and Power output)
N_turbine = 9
M_turbine = 8
NM = N_turbine * M_turbine
q_Betz_limit = (1/3) * np.ones((N_turbine, M_turbine))
q = q_Betz_limit
qs = np.reshape(q, -1)
P = np.ones((N_turbine, M_turbine))
Ps = np.reshape(P, -1)
delta_u = np.ones((N_turbine, M_turbine))
delta_u_s = np.reshape(delta_u, -1)
U_wind = 10
k = 0.04
rho_air = 1.225

R_rotor = 41.2
dx = 481.706
dy = 858.562
angl = 135
A_rotor = np.pi * R_rotor**2
[x, y] = wind_farm(angl, N_turbine, M_turbine)
xs = np.reshape(x, -1)
ys = np.reshape(y, -1)
[ovl] = overlap(N_turbine, M_turbine, R_rotor, k, xs, ys)
delta_u_i = np.zeros(N_turbine)
p_tot_new = 10
p_tot_old = 0
p_tot_old_2 = 0
it = 0
fileID = open('celldata.dat', 'w')
it = 0

while (p_tot_new - p_tot_old_2) > 0.0001:
    p_tot_old_2 = p_tot_new
    itt = it
    mar = 0

    for l in range(1, NM + 1):
        p_tot_old = 0
        mar = mar + 1

        while (p_tot_new - p_tot_old) > 0.0001:
            p_tot_old = p_tot_new

            if qs[l - 1] < 1:
                qs[l - 1] = qs[l - 1] * 1.0001

            [delta_u_s] = velocity_def(N_turbine, M_turbine, R_rotor, k, xs, qs, ovl)
            [v, P] = power_turb(U_wind, A_rotor, rho_air, delta_u_s, qs)
            p_tot_new = np.sum(P)
            fileID.write(f"{it} {mar} {p_tot_new}\n")

        it += 1

    # Marching along the points

Ps = np.reshape(P, -1)
zita = Ps / P_betz - 1

# Plotting Optimized Power output in south-east wind direction
plt.figure(3)
plt.scatter(xs, ys, 2000 * np.abs(zita))

ValueError: too many values to unpack (expected 1)